
Create the held-out labelled multimodal pool from the final 2,200-study sample.

The pool contains:

- Structured clinical features
- Processed chest radiographs
- Processed radiology reports
- Disease labels

### Output

`test_calibration_pool.csv`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

base_path = '/content/drive/MyDrive/dissertation_project/data'

processed_path = f'{base_path}/processed'
raw_path = f'{base_path}/raw'

image_download_path = f'{raw_path}/selected_images'

os.makedirs(processed_path, exist_ok=True)

print("Base path:", base_path)
print("Processed path:", processed_path)
print("Image path:", image_download_path)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Load the structured dataset
structured_file = (
    f'{processed_path}/structured_processed.csv'
)

structured_data = pd.read_csv(
    structured_file
)

print(
    "Structured dataset shape:",
    structured_data.shape
)

print(
    "\nUnique studies:",
    structured_data['study_id'].nunique()
)

print(
    "\nColumns:"
)

print(
    structured_data.columns.tolist()
)

In [ ]:
# Check the seven labels
label_columns = [
    'No Finding',
    'Support Devices',
    'Pleural Effusion',
    'Lung Opacity',
    'Atelectasis',
    'Cardiomegaly',
    'Edema'
]

available_labels = [
    col
    for col in label_columns
    if col in structured_data.columns
]

print(
    "Available labels:"
)

print(
    available_labels
)

In [ ]:
# Load the processed text
text_file = (
    f'{processed_path}/text_processed.csv'
)

text_data = pd.read_csv(
    text_file
)

print(
    "Text dataset shape:",
    text_data.shape
)

print(
    "Unique studies:",
    text_data['study_id'].nunique()
)

print(
    "\nText columns:"
)

print(
    text_data.columns.tolist()
)

In [ ]:
# Check text availability
text_data['text_available'] = (
    text_data['report_text']
    .fillna('')
    .str.strip()
    .ne('')
)

print(
    "Reports with usable text:",
    text_data['text_available'].sum()
)

print(
    "Reports without usable text:",
    (~text_data['text_available']).sum()
)

In [ ]:
# Load the processed image array
image_file = (
    f'{processed_path}/processed_images.npz'
)

image_data = np.load(
    image_file
)

processed_image_ids = (
    image_data['study_ids']
    .astype(str)
)

print(
    "Processed image array:",
    image_data['images'].shape
)

print(
    "Number of processed image studies:",
    len(processed_image_ids)
)

print(
    "Unique image studies:",
    len(np.unique(processed_image_ids))
)

In [ ]:
# Create the complete multimodal study list
structured_ids = set(
    structured_data['study_id']
    .astype(str)
)

text_ids = set(
    text_data[
        text_data['text_available']
    ]['study_id']
    .astype(str)
)

image_ids = set(
    processed_image_ids
)

complete_ids = (
    structured_ids
    & text_ids
    & image_ids
)

print(
    "Structured studies:",
    len(structured_ids)
)

print(
    "Text studies:",
    len(text_ids)
)

print(
    "Image studies:",
    len(image_ids)
)

print(
    "\nComplete multimodal studies:",
    len(complete_ids)
)

In [ ]:
# Filter the structured dataset
complete_ids = sorted(
    list(complete_ids)
)

multimodal_structured = structured_data[
    structured_data['study_id']
    .astype(str)
    .isin(complete_ids)
].copy()

print(
    "Complete structured records:",
    multimodal_structured.shape
)

print(
    "Unique studies:",
    multimodal_structured['study_id'].nunique()
)

In [ ]:
# Attach the report text
text_columns = [
    'study_id',
    'report_text'
]

text_subset = text_data[
    [
        col
        for col in text_columns
        if col in text_data.columns
    ]
].copy()

text_subset['study_id'] = (
    text_subset['study_id']
    .astype(str)
)

multimodal_structured['study_id'] = (
    multimodal_structured['study_id']
    .astype(str)
)

multimodal_pool = multimodal_structured.merge(
    text_subset,
    on='study_id',
    how='inner'
)

print(
    "After adding text:",
    multimodal_pool.shape
)

In [ ]:
# Verify the image linkage
image_id_set = set(
    processed_image_ids
)

multimodal_pool['image_available'] = (
    multimodal_pool['study_id']
    .isin(image_id_set)
)

print(
    multimodal_pool[
        'image_available'
    ].value_counts()
)

In [ ]:
# Verify text linkage
multimodal_pool['text_available'] = (
    multimodal_pool['report_text']
    .fillna('')
    .str.strip()
    .ne('')
)

print(
    multimodal_pool[
        'text_available'
    ].value_counts()
)

In [ ]:
# Keep only complete records
complete_multimodal_pool = multimodal_pool[
    (
        multimodal_pool['image_available']
        == True
    )
    &
    (
        multimodal_pool['text_available']
        == True
    )
].copy()

complete_multimodal_pool = (
    complete_multimodal_pool
    .drop_duplicates(
        subset='study_id'
    )
    .reset_index(drop=True)
)

print(
    "Complete multimodal pool:",
    len(complete_multimodal_pool)
)

print(
    "Unique studies:",
    complete_multimodal_pool[
        'study_id'
    ].nunique()
)

In [ ]:
# Confirm that 687 records can be created
TARGET_SIZE = 687

available_records = len(
    complete_multimodal_pool
)

print(
    "Available complete records:",
    available_records
)

print(
    "Required held-out records:",
    TARGET_SIZE
)

if available_records < TARGET_SIZE:

    raise ValueError(
        f"Only {available_records} complete multimodal "
        f"records are available. At least {TARGET_SIZE} "
        f"are required."
    )

print(
    "\n✓ Enough complete records are available."
)

In [ ]:
# Create the 687-record pool
test_calibration_pool = (
    complete_multimodal_pool
    .sample(
        n=TARGET_SIZE,
        random_state=42
    )
    .reset_index(drop=True)
)

print(
    "Created held-out pool:",
    test_calibration_pool.shape
)

print(
    "Unique studies:",
    test_calibration_pool[
        'study_id'
    ].nunique()
)

In [ ]:
# Check disease labels
print(
    "Disease distribution in the 687-record pool:"
)

for label in available_labels:

    positive = (
        test_calibration_pool[label]
        == 1.0
    ).sum()

    percentage = (
        positive
        / len(test_calibration_pool)
        * 100
    )

    print(
        f"{label:<25} "
        f"{positive:>4} "
        f"({percentage:>6.2f}%)"
    )

In [ ]:
# Check the held-out pool
print(
    "Number of records:",
    len(test_calibration_pool)
)

print(
    "Unique studies:",
    test_calibration_pool[
        'study_id'
    ].nunique()
)

print(
    "Duplicate studies:",
    test_calibration_pool[
        'study_id'
    ].duplicated()
    .sum()
)

print(
    "Missing report text:",
    test_calibration_pool[
        'report_text'
    ]
    .isna()
    .sum()
)

print(
    "Missing image availability:",
    test_calibration_pool[
        'image_available'
    ]
    .isna()
    .sum()
)

In [ ]:
# Save the 687-record file
output_file = (
    f'{processed_path}/test_calibration_pool.csv'
)

test_calibration_pool.to_csv(
    output_file,
    index=False
)

print(
    "Saved:",
    output_file
)

print(
    "Shape:",
    test_calibration_pool.shape
)

In [ ]:
# Reload and verify
check = pd.read_csv(
    output_file
)

print(
    "Reloaded shape:",
    check.shape
)

print(
    "Unique studies:",
    check['study_id'].nunique()
)

print(
    "\nColumns:"
)

print(
    check.columns.tolist()
)

display(
    check.head()
)

In [ ]:
# Save the IDs separately
heldout_ids_file = (
    f'{processed_path}/test_calibration_ids.csv'
)

test_calibration_pool[
    ['study_id']
].to_csv(
    heldout_ids_file,
    index=False
)

print(
    "Saved held-out study IDs:",
    heldout_ids_file
)

In [ ]:
#